# Lecture 5 - Notebook 1: Data Audit und Cleaning

**Kurs:** Data Science - Vorlesung 5  
**Datei:** `lecture_05_notebook_01_data_audit_cleaning.ipynb`

In diesem Notebook verwandeln wir einen kleinen Rohdatensatz Schritt für Schritt in einen **analysebereiten Datensatz**.

Der rote Faden:

1. Daten einlesen
2. Struktur prüfen
3. Probleme sichtbar machen
4. Kategorien und Werte bereinigen
5. fehlende Werte bewusst behandeln
6. bereinigte Daten speichern

**Wichtig:** Es geht nicht darum, jeden Befehl auswendig zu lernen. Es geht darum, eine wiederholbare Arbeitslogik zu verstehen.

## Für Studierende, die nicht anwesend waren

Dieses Notebook ist so geschrieben, dass Sie den Ablauf selbstständig nachvollziehen können.

**Wichtig:** In dieser Vorlesung geht es nicht darum, die Programmiersprache Python auswendig zu lernen. Die Codezellen sind Werkzeuge, um Data-Handling-Regeln anzuwenden.

Der fachliche Ablauf ist:

1. Rohdaten laden
2. Struktur prüfen
3. Datenprobleme sichtbar machen
4. Bereinigungsregeln bewusst anwenden
5. Entscheidungen im Code und in Markdown nachvollziehbar dokumentieren
6. bereinigte Daten speichern

Wenn eine Codezeile unklar ist, lesen Sie zuerst die Markdown-Erklärung direkt darüber. Danach führen Sie die Zelle aus und interpretieren das Ergebnis.

## Dateien für dieses Notebook

Dieses Notebook verwendet:

- `lecture_05_employee_survey_raw.csv` - Rohdaten
- `lecture_05_codebook.md` - Variablenbeschreibung
- erzeugt am Ende: `lecture_05_employee_survey_cleaned.csv`

Bitte öffnen Sie in VS Code den Ordner, in dem diese Dateien liegen. Der Ordner ist der Speicherort Ihrer Dateien. Der Kernel ist die Python-Umgebung, die das Notebook ausführt.

## Vor dem Start

Bitte prüfen Sie:

- VS Code ist geöffnet
- der Kursordner ist geöffnet
- der richtige Kernel ist ausgewählt
- die Datei `lecture_05_employee_survey_raw.csv` liegt im gleichen Ordner wie dieses Notebook

Wenn der Import oder das Laden der CSV-Datei nicht funktioniert, prüfen Sie zuerst den Ordner und den Kernel.

In [1]:
# Wir importieren die Bibliotheken, die wir in diesem Notebook brauchen.
# pandas ist unser Hauptwerkzeug für tabellarische Daten.
# numpy verwenden wir hier vor allem für fehlende Werte (np.nan).

import pandas as pd
import numpy as np

# Diese Anzeigeoption sorgt dafür, dass pandas mehr Spalten nebeneinander anzeigt.
# Das ist hilfreich, weil unser Datensatz mehrere Skalen- und Kontextvariablen enthält.
pd.set_option("display.max_columns", 60)

## 1. Rohdaten einlesen

Wir starten immer mit den Rohdaten.  
Die Rohdaten bleiben unverändert. Alle Bereinigungsschritte machen wir später in einer Kopie.

**Aufgabe:** Laden Sie die CSV-Datei und betrachten Sie die ersten Zeilen.

In [2]:
# Die Datei muss im gleichen Ordner liegen wie dieses Notebook.
# Falls die Datei in einem Unterordner liegt, muss der Pfad angepasst werden.
# Beispiel: pd.read_csv("data/lecture_05_employee_survey_raw.csv")

df = pd.read_csv("lecture_05_employee_survey_raw.csv")

# head() zeigt die ersten fünf Zeilen.
# Das ist meistens der schnellste erste Blick in einen neuen Datensatz.
df.head()

,participant_id,department,gender,age,weekly_hours,sleep_hours,commute_minutes,workload_1,workload_2,autonomy_1,autonomy_2,support_1,support_2,stress_score,motivation,satisfaction,performance_rating,absent_days
0,P001,HR,Male,42.0,45.4,8.6,21.0,4.0,4,3,4,3,2.0,54.0,3.3,3.2,3.1,4.0
1,P002,Operations,Male,22.0,45.3,5.8,30.0,4.0,4,4,3,5,5.0,33.0,4.0,4.1,4.5,3.0
2,P003,Marketing,Female,36.0,33.1,9.1,26.0,3.0,3,4,4,2,2.0,37.0,3.9,3.2,3.5,0.0
3,P004,sales,Female,47.0,35.3,5.0,23.0,3.0,3,4,4,2,1.0,54.0,2.8,3.2,3.4,2.0
4,P005,Sales,Female,25.0,42.6,6.8,23.0,3.0,3,3,3,2,2.0,49.0,3.8,2.9,3.3,2.0


## 2. Erste Orientierung: Zeilen, Spalten, Namen

Bevor wir irgendetwas bereinigen, beantworten wir einfache Strukturfragen:

- Wie viele Fälle gibt es?
- Wie viele Variablen gibt es?
- Wie heißen die Spalten?

In [3]:
# shape gibt ein Tupel zurück: (Anzahl Zeilen, Anzahl Spalten).
# In unserem Beispiel entspricht eine Zeile einer befragten Person.

print("Shape:", df.shape)

# columns enthält die Spaltennamen.
# Die Umwandlung in eine Liste macht die Ausgabe gut lesbar.
print("\nSpalten:")
print(df.columns.tolist())

Shape: (81, 18)

Spalten:
['participant_id', 'department', 'gender', 'age', 'weekly_hours', 'sleep_hours', 'commute_minutes', 'workload_1', 'workload_2', 'autonomy_1', 'autonomy_2', 'support_1', 'support_2', 'stress_score', 'motivation', 'satisfaction', 'performance_rating', 'absent_days']


## 3. Datentypen und fehlende Werte prüfen

`info()` zeigt:

- die Spaltennamen
- die Anzahl nicht-fehlender Werte pro Spalte
- die von pandas erkannten Datentypen

Das ist ein erster Data-Audit-Schritt.

In [4]:
# info() gibt eine kompakte Übersicht über den DataFrame.
# Achten Sie besonders auf:
# - Non-Null Count: Gibt es fehlende Werte?
# - Dtype: Hat pandas Zahlen wirklich als Zahlen erkannt?

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   participant_id      81 non-null     object 
 1   department          80 non-null     object 
 2   gender              80 non-null     object 
 3   age                 80 non-null     float64
 4   weekly_hours        81 non-null     float64
 5   sleep_hours         81 non-null     float64
 6   commute_minutes     80 non-null     float64
 7   workload_1          80 non-null     float64
 8   workload_2          81 non-null     int64  
 9   autonomy_1          81 non-null     int64  
 10  autonomy_2          81 non-null     int64  
 11  support_1           81 non-null     int64  
 12  support_2           80 non-null     float64
 13  stress_score        80 non-null     float64
 14  motivation          80 non-null     float64
 15  satisfaction        81 non-null     float64
 16  performanc

In [5]:
# isna() prüft pro Zelle, ob ein Wert fehlt.
# sum() zählt diese fehlenden Werte pro Spalte.
# sort_values() sortiert die Spalten so, dass die meisten fehlenden Werte oben stehen.

missing_values = df.isna().sum().sort_values(ascending=False)

missing_values

department            1
gender                1
absent_days           1
age                   1
workload_1            1
commute_minutes       1
motivation            1
performance_rating    1
support_2             1
stress_score          1
participant_id        0
workload_2            0
weekly_hours          0
sleep_hours           0
autonomy_1            0
support_1             0
autonomy_2            0
satisfaction          0
dtype: int64

## 4. Summary Statistics als Diagnosewerkzeug

`describe()` ist hier noch kein Ergebnisbericht.  
Es ist ein Diagnosewerkzeug.

Wir prüfen zum Beispiel:

- Gibt es negative Werte, die nicht möglich sind?
- Sind Min und Max plausibel?
- Wirken Mittelwert und Streuung ungefähr erwartbar?

In [6]:
# Wir wählen zunächst nur numerische Spalten aus.
# describe().T transponiert die Ausgabe, damit jede Variable eine eigene Zeile hat.

numeric_summary = df.select_dtypes(include="number").describe().T

numeric_summary

,count,mean,std,min,25%,50%,75%,max
age,80.0,31.937500,6.751172,17.0,26.00,33.00,36.00,51.0
weekly_hours,81.0,40.698765,6.602490,23.8,36.70,40.40,43.90,72.0
sleep_hours,81.0,7.048148,1.411215,-1.0,6.40,7.00,7.60,13.5
commute_minutes,80.0,27.462500,19.661016,3.0,12.00,22.00,35.25,81.0
workload_1,80.0,2.975000,0.927430,1.0,2.00,3.00,4.00,5.0
workload_2,81.0,3.000000,0.987421,1.0,3.00,3.00,4.00,5.0
autonomy_1,81.0,3.481481,0.776388,2.0,3.00,4.00,4.00,5.0
autonomy_2,81.0,3.444444,0.880341,1.0,3.00,3.00,4.00,5.0
support_1,81.0,3.518519,0.950146,1.0,3.00,4.00,4.00,5.0
support_2,80.0,3.462500,0.980103,1.0,3.00,3.50,4.00,5.0


## 5. Kategorien prüfen

Bei kategorialen Variablen ist `value_counts()` oft besonders wichtig.

Warum?  
Weil ein Computer `Sales`, `sales` und ` SALES ` zunächst als unterschiedliche Kategorien behandeln kann.

In [7]:
# dropna=False sorgt dafür, dass fehlende Werte ebenfalls angezeigt werden.
# Das ist für einen Data Audit wichtig.

df["department"].value_counts(dropna=False)

department
Sales         24
Marketing     14
HR            13
Operations    13
IT            13
sales          1
IT             1
MKT            1
NaN            1
Name: count, dtype: int64

In [8]:
# Auch bei gender prüfen wir die tatsächlich vorkommenden Kategorien.
# Hier geht es noch nicht um Interpretation, sondern um Datenqualität.

df["gender"].value_counts(dropna=False)

gender
Female               40
Male                 33
Prefer not to say     3
Nonbinary             2
F                     1
NaN                   1
m                     1
Name: count, dtype: int64

## 6. Arbeitskopie für die Bereinigung anlegen

Wir verändern nicht direkt `df`, sondern erstellen eine Kopie `clean`.

Das ist eine wichtige Arbeitsregel:

- `df` bleibt der Rohdatensatz
- `clean` wird der bereinigte Datensatz

In [9]:
# copy() erstellt eine unabhängige Kopie des DataFrames.
# Alle folgenden Bereinigungsschritte passieren in clean.

clean = df.copy()

# Spaltennamen werden vereinheitlicht:
# - strip() entfernt Leerzeichen am Anfang und Ende
# - lower() macht alles klein
# Das vermeidet spätere Tipp- und Schreibprobleme.
clean.columns = [c.strip().lower() for c in clean.columns]

clean.columns.tolist()

['participant_id',
 'department',
 'gender',
 'age',
 'weekly_hours',
 'sleep_hours',
 'commute_minutes',
 'workload_1',
 'workload_2',
 'autonomy_1',
 'autonomy_2',
 'support_1',
 'support_2',
 'stress_score',
 'motivation',
 'satisfaction',
 'performance_rating',
 'absent_days']

## 7. Kategorien bereinigen: department

Ziel:

- Leerzeichen entfernen
- Groß-/Kleinschreibung vereinheitlichen
- Abkürzungen auflösen
- fehlende Werte sichtbar markieren

**Wichtig:** Die Zuordnung ist eine inhaltliche Entscheidung. Der Code dokumentiert diese Entscheidung.

In [10]:
# Zuerst machen wir alle Einträge zu Strings.
# Danach entfernen wir Leerzeichen und vereinheitlichen die Groß-/Kleinschreibung.
clean["department"] = clean["department"].astype("string").str.strip().str.lower()

# Jetzt führen wir inhaltlich gleiche Kategorien zusammen.
# Beispiel: "mkt" und "marketing" werden beide zu "Marketing".
department_map = {
    "sales": "Sales",
    "hr": "HR",
    "it": "IT",
    "mkt": "Marketing",
    "marketing": "Marketing",
    "operations": "Operations",
}

clean["department"] = clean["department"].replace(department_map)

# Fehlende Werte in department markieren wir als "Unknown".
# Das ist hier sinnvoll, weil department eine Kategorie ist.
clean["department"] = clean["department"].fillna("Unknown")

clean["department"].value_counts(dropna=False)

department
Sales         25
Marketing     15
IT            14
HR            13
Operations    13
Unknown        1
Name: count, dtype: Int64

## 8. Kategorien bereinigen: gender

Wir gehen nach derselben Logik vor:

1. als String behandeln
2. Leerzeichen entfernen
3. klein schreiben
4. inhaltlich gleiche Kategorien zusammenführen
5. fehlende Werte markieren

In [11]:
clean["gender"] = clean["gender"].astype("string").str.strip().str.lower()

gender_map = {
    "f": "Female",
    "female": "Female",
    "m": "Male",
    "male": "Male",
    "nonbinary": "Nonbinary",
    "prefer not to say": "Prefer not to say",
    "": "Unknown",
}

clean["gender"] = clean["gender"].replace(gender_map)
clean["gender"] = clean["gender"].fillna("Unknown")

clean["gender"].value_counts(dropna=False)

gender
Female               41
Male                 34
Prefer not to say     3
Nonbinary             2
Unknown               1
Name: count, dtype: Int64

## 9. Doppelte Fälle prüfen

Wenn eine Person doppelt im Datensatz steht, zählt sie in Analysen doppelt.  
Deshalb prüfen wir die ID-Spalte.

In [12]:
# duplicated(..., keep=False) zeigt alle Zeilen, deren participant_id mehrfach vorkommt.
# sort_values() sortiert die Ausgabe nach ID, damit doppelte Fälle nebeneinander stehen.

duplicate_rows = clean[clean.duplicated(subset="participant_id", keep=False)].sort_values("participant_id")

duplicate_rows

,participant_id,department,gender,age,weekly_hours,sleep_hours,commute_minutes,workload_1,workload_2,autonomy_1,autonomy_2,support_1,support_2,stress_score,motivation,satisfaction,performance_rating,absent_days
9,P010,Marketing,Female,32.0,45.7,6.5,12.0,2.0,3,4,5,4,3.0,31.0,3.6,4.6,3.9,1.0
80,P010,Marketing,Female,32.0,45.7,6.5,12.0,2.0,3,4,5,4,3.0,31.0,3.6,2.0,3.9,1.0


In [13]:
# Für diese Demonstration behalten wir den ersten Eintrag pro participant_id.
# In echten Projekten würde man häufig vorher prüfen, welcher Eintrag korrekt ist.

clean = clean.drop_duplicates(subset="participant_id", keep="first").reset_index(drop=True)

print("Shape nach Entfernen doppelter IDs:", clean.shape)

Shape nach Entfernen doppelter IDs: (80, 18)


## 10. Numerische Spalten als Zahlen interpretieren

Manchmal sehen Werte wie Zahlen aus, sind aber technisch als Text gespeichert.  
`pd.to_numeric(..., errors="coerce")` versucht, Werte in Zahlen umzuwandeln.

Wenn etwas nicht umwandelbar ist, wird es zu `NaN`.

In [14]:
numeric_cols = [
    "age", "weekly_hours", "sleep_hours", "commute_minutes",
    "workload_1", "workload_2", "autonomy_1", "autonomy_2",
    "support_1", "support_2", "stress_score", "motivation",
    "satisfaction", "performance_rating", "absent_days",
]

for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean[numeric_cols].dtypes

age                   float64
weekly_hours          float64
sleep_hours           float64
commute_minutes       float64
workload_1            float64
workload_2              int64
autonomy_1              int64
autonomy_2              int64
support_1               int64
support_2             float64
stress_score          float64
motivation            float64
satisfaction          float64
performance_rating    float64
absent_days           float64
dtype: object

## 11. Gültige Wertebereiche festlegen

Ein Wert kann vorhanden sein und trotzdem unplausibel sein.

Beispiele:

- `sleep_hours = -1` ist nicht plausibel
- Likert-Skalen sollten zwischen 1 und 5 liegen
- `stress_score` sollte zwischen 0 und 100 liegen

Wir legen Regeln fest und setzen Werte außerhalb des Bereichs auf `NaN`.

In [15]:
# rules ist ein Dictionary.
# Jede Variable bekommt einen unteren und oberen gültigen Wert.
# Diese Regeln sind Teil unserer Datenentscheidung und sollten dokumentiert werden.

rules = {
    "age": (18, 67),
    "weekly_hours": (0, 60),
    "sleep_hours": (0, 12),
    "commute_minutes": (0, 180),
    "stress_score": (0, 100),
    "motivation": (1, 5),
    "satisfaction": (1, 5),
    "performance_rating": (1, 5),
    "absent_days": (0, 60),
    "workload_1": (1, 5),
    "workload_2": (1, 5),
    "autonomy_1": (1, 5),
    "autonomy_2": (1, 5),
    "support_1": (1, 5),
    "support_2": (1, 5),
}

# Wir wenden die Regeln nacheinander an.
# between(lo, hi) prüft, ob ein Wert im gültigen Bereich liegt.
# Das ~ dreht die Bedingung um: Werte NICHT im gültigen Bereich.
for col, (lo, hi) in rules.items():
    invalid_mask = ~clean[col].between(lo, hi)
    clean.loc[invalid_mask, col] = np.nan

# Nach dem Markieren unplausibler Werte prüfen wir erneut die fehlenden Werte.
clean.isna().sum().sort_values(ascending=False).head(15)

sleep_hours           2
age                   2
performance_rating    1
absent_days           1
weekly_hours          1
commute_minutes       1
workload_1            1
satisfaction          1
support_2             1
stress_score          1
motivation            1
participant_id        0
department            0
gender                0
autonomy_1            0
dtype: int64

## 12. Fehlende Werte behandeln

Für diese Demonstration verwenden wir eine einfache Strategie:

- numerische fehlende Werte werden durch den Median der jeweiligen Spalte ersetzt

Warum Median?

- leicht verständlich
- robust gegenüber Ausreißern
- für eine erste Übung gut nachvollziehbar

In echten Projekten muss die Strategie immer begründet werden.

In [16]:
# Wir gehen alle numerischen Spalten durch.
# Wenn Werte fehlen, ersetzen wir sie durch den Median derselben Spalte.

for col in numeric_cols:
    median_value = clean[col].median()
    clean[col] = clean[col].fillna(median_value)

# Kontrollcheck:
# Wenn hier 0 steht, gibt es keine fehlenden Werte mehr.
total_missing = clean.isna().sum().sum()

print("Fehlende Werte insgesamt:", total_missing)

Fehlende Werte insgesamt: 0


## 13. Analysebereiten Datensatz speichern

Jetzt speichern wir den bereinigten Datensatz als neue CSV-Datei.

Wichtig:

- Die Rohdaten bleiben erhalten.
- Die bereinigte Datei ist der Ausgangspunkt für Notebook 2.

In [17]:
# index=False verhindert, dass pandas den DataFrame-Index als zusätzliche Spalte speichert.

clean.to_csv("lecture_05_employee_survey_cleaned.csv", index=False)

clean.head()

,participant_id,department,gender,age,weekly_hours,sleep_hours,commute_minutes,workload_1,workload_2,autonomy_1,autonomy_2,support_1,support_2,stress_score,motivation,satisfaction,performance_rating,absent_days
0,P001,HR,Male,42.0,45.4,8.6,21.0,4.0,4.0,3.0,4.0,3.0,2.0,54.0,3.3,3.2,3.1,4.0
1,P002,Operations,Male,22.0,45.3,5.8,30.0,4.0,4.0,4.0,3.0,5.0,5.0,33.0,4.0,4.1,4.5,3.0
2,P003,Marketing,Female,36.0,33.1,9.1,26.0,3.0,3.0,4.0,4.0,2.0,2.0,37.0,3.9,3.2,3.5,0.0
3,P004,Sales,Female,47.0,35.3,5.0,23.0,3.0,3.0,4.0,4.0,2.0,1.0,54.0,2.8,3.2,3.4,2.0
4,P005,Sales,Female,25.0,42.6,6.8,23.0,3.0,3.0,3.0,3.0,2.0,2.0,49.0,3.8,2.9,3.3,2.0


## Abschluss-Check

Nach diesem Notebook sollten Sie erklären können:

- was `df.head()`, `df.shape`, `df.info()` und `describe()` leisten
- warum wir Kategorien vereinheitlichen
- warum IDs auf Dubletten geprüft werden
- warum unplausible Werte nicht einfach ignoriert werden
- warum die Bereinigung im Code dokumentiert sein sollte

Die Datei `lecture_05_employee_survey_cleaned.csv` wird im nächsten Notebook verwendet.